# Ecommerce ETL Project

## 项目介绍

本项目基于 Pandas 对电商销售数据进行 ETL 数据处理与分析。

项目使用 Online Retail 数据集，对原始订单数据进行：

- 数据读取（Extract）
- 数据清洗（Cleaning）
- 数据转换（Transform）
- 数据聚合（Aggregation）
- 结果表输出（Load）

在项目中，使用 Pandas 完成了：

- 缺失值处理
- 时间字段处理
- 销售额字段构建
- groupby 聚合统计
- Excel 结果表输出

最终生成了多个业务结果表，例如：

- 月销售额表
- 国家销售汇总表
- 商品销售汇总表
- 客户消费汇总表

## 使用技术

- Python
- Pandas
- NumPy
- Jupyter Notebook
- openpyxl

## 数据集

Online Retail Dataset（Kaggle 电商订单数据集）

## 导入库

In [77]:
#from curses.ascii import isdigit

import pandas as pd
import numpy as np

## 数据读取与初步探索

In [78]:
df = pd.read_excel("data/raw/online_retail_II.xlsx")
df.head(10)
#发票号    库存代码                         商品描述     数量          发票日期  单价   客户ID          国家/地区

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [79]:
#数据概览
df.info()   #object 类型通常表示字符串或混合类型的列

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [80]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845017856,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


In [81]:
df.shape

(525461, 8)

In [82]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

## 数据清洗

In [83]:
df.isna().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

## 数据清洗与转化

In [84]:
#删除缺失值
df.dropna(inplace=True)

In [85]:
#查看、删除重复值
df.duplicated().sum()
df.drop_duplicates(inplace=True)
#default =  df['Invoice'].str.extract(r'c(\d+)').len()

In [86]:
#查看、删除库存代码字段重复值、错误值
df["StockCode"].duplicated().sum()

np.int64(406732)

In [87]:
#df['StockCode'].drop_duplicates(inplace=True)

In [106]:
df['Description'] = df['Description'].str.strip()

In [107]:
df = df[df['StockCode'].str.match(r"^\d{5}$",na=False)]

In [108]:
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

In [109]:
#去除购买数量、单价错误值 脏数据
df = df[(df['Quantity']>0)  & (df['Price']>0)]

In [110]:
#修改字段类型
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Country'] = df['Country'].astype('category')

In [111]:
df['sale'] = df['Quantity']*df['Price']
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

In [120]:
#断言 数据质量检查（Data Quality Check）
assert (df['sale'] >= 0).all()
assert (df['Quantity'] > 0).all()
assert (df['Customer ID'].notna().all())

## 数据聚合与分析

In [113]:
#按维度生成结果表
'''
df.groupby('Description').agg({
    'sale' : ['sum','mean','max'],
    'Customer ID' : ['nunique','count',]
}).reset_index().to_csv('output/product_summary.csv',index=False)
'''

"\ndf.groupby('Description').agg({\n    'sale' : ['sum','mean','max'],\n    'Customer ID' : ['nunique','count',]\n}).reset_index().to_csv('output/product_summary.csv',index=False)\n"

In [114]:
def export_excel(df:pd.DataFrame,path:str):
    """
    导出 Excel 文件
    :param df:   导出数据
    :param path: 导出文件路径
    :return: NULL
    """
    df = df.round(2)
    df.to_excel(path,index=False)

    print(f'文件已导出: {path}')

In [115]:
#商品总结表
product_summary = df.groupby('Description').agg({
    'sale': ['sum', 'mean', 'max'],
    'Customer ID': ['nunique', 'count']
}).reset_index()

product_summary.columns = [
    'Description',
    'sale_sum',
    'sale_mean',
    'sale_max',
    'customer_nunique',
    'customer_count'
]

export_excel(product_summary,"output/product_summary.xlsx")

#product_summary = product_summary.round(2)

#result.to_excel('output/product_summary.xlsx',index=False)

文件已导出: output/product_summary.xlsx


In [116]:
#月销售额表
monthly_sales = df.groupby('YearMonth').agg({
    'sale' : ['sum','mean','count'],
    'Description' : 'nunique',
    'Customer ID' : ['nunique','count']
}).reset_index()

monthly_sales.columns=[
    'YearMonth','sale_sum','sale_mean','sale_count',
    'Description_nunique',
    'customer_nunique','customer_count'
]

export_excel(monthly_sales,'output/monthly_sales.xlsx')

#monthly_sales = monthly_sales.round(2)
#monthly_sales.to_excel('output/monthly_sales.xlsx',index=False)

文件已导出: output/monthly_sales.xlsx


In [117]:
#国家销售表
country_sales = df.groupby('Country',observed=False).agg({
    'sale' : ['sum','mean','max'],
    'Customer ID' : ['nunique','count'],
    'Description' : 'nunique'
}).reset_index()

country_sales.columns = [
    'Country',
    'sale_sum','sale_mean','sale_max',
    'customer_nunique','customer_count',
    'Description_count'
]

export_excel(country_sales,'output/country_sales.xlsx')

#country_sales = country_sales.round(2)
#country_sales.to_excel('output/country_sales.xlsx',index=False)

文件已导出: output/country_sales.xlsx


In [118]:
#客户销量表
customer_sales = df.groupby('Customer ID',observed=False).agg({
    'sale' : ['sum','mean','max'],
    'Description' : 'nunique'
}).reset_index()

customer_sales.columns = [
    'Customer ID',
    'sale_sum','sale_mean','sale_max',
    'Description_nunique'
]


export_excel(customer_sales,'output/customer_sales.xlsx')

#customer_sales = customer_sales.round(2)
#customer_sales.to_excel('output/customer_sales.xlsx',index=False)

文件已导出: output/customer_sales.xlsx


✅ 数据读取
✅ 数据清洗
✅ 数据转换
✅ groupby 聚合
✅ 结果表输出